# 🏆 DATATHON 2026 — EDA: Stage 2 (Behavior) + Stage 3 (Conversion)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib.ticker as mticker, seaborn as snsimport warnings; warnings.filterwarnings('ignore')plt.rcParams.update({'figure.figsize':(14,6),'figure.dpi':120,'figure.facecolor':'#0d1117','axes.facecolor':'#161b22','axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9','axes.grid':True,'grid.color':'#21262d','grid.alpha':0.6,'text.color':'#c9d1d9','xtick.color':'#8b949e','ytick.color':'#8b949e','font.size':11,'axes.titlesize':14,'legend.facecolor':'#161b22','legend.edgecolor':'#30363d'})PALETTE = ['#58a6ff','#f78166','#3fb950','#d2a8ff','#f0883e','#79c0ff','#56d364']sns.set_palette(PALETTE)import sys; sys.path.insert(0, '..')from joined.data_pipeline import load_transaction_master, load_daily_summary, load_returns_enriched, load_reviews_enrichedtxn = load_transaction_master(); daily = load_daily_summary(); ret = load_returns_enriched(); rev = load_reviews_enriched()orders = pd.read_csv('../dataset_cleaned/orders.csv', parse_dates=['order_date'])inventory = pd.read_csv('../dataset_cleaned/inventory.csv', parse_dates=['snapshot_date'])txn['order_date'] = pd.to_datetime(txn['order_date'])print("✅ Data loaded")

---# 🧠 Giai đoạn 2: Hành vi & Sở thích (Behavior & Preferences)> **Diagnostic:** Ai đang mua gì? Tồn kho có đáp ứng được nhu cầu?

### 2.1 Chân dung khách hàng — Age Group × Segment

In [ ]:
delivered = txn[txn['order_status'].isin(['delivered','shipped'])]persona = delivered.groupby(['age_group','segment'])['line_revenue'].sum().reset_index()persona_pivot = persona.pivot(index='age_group', columns='segment', values='line_revenue').fillna(0)fig, ax = plt.subplots(figsize=(12, 6))sns.heatmap(persona_pivot/1e6, annot=True, fmt=',.0f', cmap='Blues', ax=ax, linewidths=0.5)ax.set_title('👤 Revenue Heatmap: Nhóm tuổi × Phân khúc (Triệu VND)', fontsize=14, fontweight='bold')plt.tight_layout(); plt.show()top = persona.sort_values('line_revenue', ascending=False).iloc[0]print(f"🏆 Phân khúc lớn nhất: {top['age_group']} × {top['segment']} = {top['line_revenue']/1e9:.1f}B VND")

### 2.2 Phân bổ doanh thu theo Danh mục × Giới tính

In [ ]:
cat_gender = delivered.groupby(['category','gender'])['line_revenue'].sum().reset_index()cat_pivot = cat_gender.pivot(index='category', columns='gender', values='line_revenue').fillna(0)cat_pivot = cat_pivot.sort_values(cat_pivot.columns.tolist(), ascending=False)fig, ax = plt.subplots(figsize=(14, 6))cat_pivot.plot(kind='barh', stacked=True, ax=ax, color=PALETTE[:3], alpha=0.85)ax.set_title('🛍️ Doanh thu theo Danh mục × Giới tính', fontweight='bold')ax.set_xlabel('Revenue (VND)')ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.1f}B'))ax.legend(title='Giới tính', framealpha=0.8)plt.tight_layout(); plt.show()

### 2.3 ⚠️ Sức khỏe Tồn kho (Inventory Health Index)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))# Distributionaxes[0].hist(inventory['inventory_health_index'], bins=40, color=PALETTE[0], alpha=0.7, edgecolor='white')axes[0].axvline(inventory['inventory_health_index'].mean(), color=PALETTE[1], linestyle='--', linewidth=2,                label=f"Mean={inventory['inventory_health_index'].mean():.1f}")axes[0].set_title('Phân phối Health Index', fontweight='bold')axes[0].legend()# By segmentinv_seg = inventory.groupby('segment')['inventory_health_index'].mean().sort_values()axes[1].barh(inv_seg.index, inv_seg.values, color=[PALETTE[1] if v<50 else PALETTE[2] for v in inv_seg])axes[1].set_title('Health Index TB theo Segment', fontweight='bold')axes[1].axvline(50, color='#8b949e', linestyle='--', alpha=0.5)for i, v in enumerate(inv_seg.values):    axes[1].text(v+0.5, i, f'{v:.1f}', va='center', fontsize=10, color='#c9d1d9')# Dual flag anomaly over timedual = inventory.groupby(['year','month'])['dual_flag_anomaly'].sum().reset_index()dual['period'] = dual['year'].astype(str) + '-' + dual['month'].astype(str).str.zfill(2)axes[2].bar(range(len(dual)), dual['dual_flag_anomaly'], color=PALETTE[1], alpha=0.7)axes[2].set_title('Stockout+Overstock đồng thời/tháng', fontweight='bold')axes[2].set_xlabel('Tháng')step = max(1, len(dual)//10)axes[2].set_xticks(range(0, len(dual), step))axes[2].set_xticklabels(dual['period'].iloc[::step], rotation=45, fontsize=8)plt.tight_layout(); plt.show()pct_danger = (inventory['inventory_health_index'] < 30).mean() * 100print(f"🔴 {pct_danger:.1f}% sản phẩm-tháng có Health Index < 30 (Nguy hiểm)")print(f"🔴 {inventory['dual_flag_anomaly'].sum():,} trường hợp vừa hết hàng vừa tồn kho quá tải")

---# 💳 Giai đoạn 3: Chuyển đổi & Vận hành (Conversion & Fulfillment)> **Diagnostic + Prescriptive:** Nút thắt ở đâu? Rò rỉ doanh thu bao nhiêu?

### 3.1 Hiệu suất theo Thiết bị (Device Type)

In [ ]:
device = orders.groupby('device_type').agg(    total=('order_id','count'),    delivered=('order_status', lambda x: (x=='delivered').sum()),    cancelled=('order_status', lambda x: (x=='cancelled').sum())).reset_index()device['delivery_rate'] = device['delivered'] / device['total'] * 100device['cancel_rate'] = device['cancelled'] / device['total'] * 100fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))ax1.bar(device['device_type'], device['total'], color=PALETTE[:len(device)], alpha=0.8)ax1.set_title('Số đơn hàng theo Thiết bị', fontweight='bold')for i, v in enumerate(device['total']): ax1.text(i, v*1.02, f'{v:,}', ha='center', fontsize=10, color='#c9d1d9')x = range(len(device))w = 0.35ax2.bar([i-w/2 for i in x], device['delivery_rate'], w, label='Delivered %', color=PALETTE[2])ax2.bar([i+w/2 for i in x], device['cancel_rate'], w, label='Cancelled %', color=PALETTE[1])ax2.set_xticks(x); ax2.set_xticklabels(device['device_type'])ax2.set_title('Tỷ lệ Giao/Hủy theo Thiết bị', fontweight='bold')ax2.legend()plt.tight_layout(); plt.show()

### 3.2 Giao hàng chậm → Trả hàng & Đánh giá thấp?

In [ ]:
ship_txn = txn[txn['delivery_days'].notna()].copy()ship_txn['delivery_bucket'] = pd.cut(ship_txn['delivery_days'], bins=[0,3,7,14,30,999],                                      labels=['1-3d','4-7d','8-14d','15-30d','>30d'])# Return rate by delivery speedret_by_speed = ship_txn.merge(ret[['order_id','product_id']].assign(returned=1),                               on=['order_id','product_id'], how='left')ret_by_speed['returned'] = ret_by_speed['returned'].fillna(0)speed_agg = ret_by_speed.groupby('delivery_bucket').agg(    return_rate=('returned','mean'), count=('order_id','count')).reset_index()# Rating by delivery speedrev_merge = ship_txn.merge(rev[['order_id','product_id','rating']], on=['order_id','product_id'], how='inner')rating_agg = rev_merge.groupby('delivery_bucket')['rating'].mean().reset_index()fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))colors = [PALETTE[2], PALETTE[0], PALETTE[3], PALETTE[1], '#ff4444']ax1.bar(speed_agg['delivery_bucket'], speed_agg['return_rate']*100, color=colors[:len(speed_agg)])ax1.set_title('Tỷ lệ Trả hàng theo Thời gian Giao', fontweight='bold')ax1.set_ylabel('Return Rate (%)')ax2.bar(rating_agg['delivery_bucket'], rating_agg['rating'], color=colors[:len(rating_agg)])ax2.set_title('Rating TB theo Thời gian Giao', fontweight='bold')ax2.set_ylabel('Rating (1-5)')ax2.axhline(rev['rating'].mean(), color='#8b949e', linestyle='--', alpha=0.5, label=f"Overall={rev['rating'].mean():.2f}")ax2.legend()plt.tight_layout(); plt.show()

### 3.3 ⚠️ Khủng hoảng Refund Leakage (59K đơn chưa hoàn tiền)

In [ ]:
unref = orders[orders['is_unrefunded'] == 1].copy()payments = pd.read_csv('../dataset_cleaned/payments.csv')unref_pay = unref.merge(payments, on='order_id', how='left')total_stuck = unref_pay['payment_value'].sum()unref['year'] = unref['order_date'].dt.yearunref_trend = unref.groupby('year').size().reset_index(name='count')fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))ax1.bar(unref_trend['year'], unref_trend['count'], color=PALETTE[1], alpha=0.8)ax1.set_title(f'Đơn chưa hoàn tiền theo Năm (Tổng: {len(unref):,})', fontweight='bold')ax1.set_ylabel('Số đơn')# By payment methodpm = unref.groupby('payment_method').size().sort_values(ascending=True)ax2.barh(pm.index, pm.values, color=PALETTE[0], alpha=0.8)ax2.set_title('Phương thức thanh toán', fontweight='bold')for i, v in enumerate(pm.values):    ax2.text(v+100, i, f'{v:,}', va='center', fontsize=10, color='#c9d1d9')plt.tight_layout(); plt.show()print(f"💰 Tổng tiền bị 'kẹt': {total_stuck/1e9:,.1f} tỷ VND")print(f"🔴 Prescriptive: Cần quy trình Auto-Refund trong 7 ngày cho đơn cancelled")

### 3.4 AOV (Average Order Value) theo thời gian

In [ ]:
order_value = txn.groupby(['order_id','order_date']).agg(order_rev=('line_revenue','sum')).reset_index()order_value['order_date'] = pd.to_datetime(order_value['order_date'])order_value['year_month'] = order_value['order_date'].dt.to_period('M')aov = order_value.groupby('year_month')['order_rev'].mean().reset_index()aov['year_month'] = aov['year_month'].dt.to_timestamp()fig, ax = plt.subplots(figsize=(14, 5))ax.plot(aov['year_month'], aov['order_rev']/1e3, color=PALETTE[0], linewidth=2)ax.fill_between(aov['year_month'], aov['order_rev']/1e3, alpha=0.15, color=PALETTE[0])ax.set_title('💰 AOV (Average Order Value) theo tháng', fontweight='bold')ax.set_ylabel('AOV (Nghìn VND)')ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}K'))plt.tight_layout(); plt.show()print(f"AOV trung bình: {order_value['order_rev'].mean()/1e3:,.0f}K VND")